# Kaito Model — Training Notebook

This notebook walks through the full training pipeline:
1. Setup & imports
2. Data loading (local file or HuggingFace dataset)
3. Model initialisation
4. Training
5. Text generation

## 1. Setup

In [ ]:
import torch
from main import kaitomodel
from data_prep.preprocess_text import PreprocessText
from text_generation import generate_text
from train import train_model
from config import *
import tiktoken

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = tiktoken.get_encoding("gpt2")

## 2. Data Loading

Choose one of the options below.

### Option A: Single .txt file (default — the-verdict.txt)

In [ ]:
# Uses the default file: data_prep/the-verdict.txt
preprocessor = PreprocessText()
train_loader, val_loader = preprocessor.preprocess()
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

### Option B: Custom .txt file

```python
preprocessor = PreprocessText("path/to/your-corpus.txt")
train_loader, val_loader = preprocessor.preprocess()
```

### Option C: HuggingFace dataset (e.g., OpenWebText, WikiText-103)

```python
# Requires: pip install datasets
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

class StreamingTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=MAX_LENGTH, stride=STRIDE):
        self.input_ids = []
        self.target_ids = []
        for text in texts:
            token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
            for i in range(0, len(token_ids) - max_length, stride):
                self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
                self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# Load 5000 articles from OpenWebText (streaming to save RAM)
ds = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
texts = [next(ds)["text"] for _ in range(5000)]

full_dataset = StreamingTextDataset(texts, tokenizer)
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
```

## 3. Model Initialisation

In [ ]:
model = kaitomodel()
model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model has {total_params:,} parameters")
print(f"Dtype: {next(model.parameters()).dtype}")

## 4. Training

The training loop internally creates:
- **AdamW** optimizer with decoupled weight decay (no decay on biases/norms)
- **Linear warmup + cosine decay** learning rate scheduler
- **Gradient accumulation** (effective batch = BATCH_SIZE × ACCUMULATION_STEPS)
- **Gradient clipping** at GRAD_CLIP_MAX_NORM=1.0
- **Z-loss** auxiliary term for logit stabilisation

In [ ]:
num_epochs = 3

train_losses, val_losses, track_tokens_seen = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=num_epochs,
    eval_freq=5,
    eval_iter=5,
    start_context="The meaning of life is",
    tokenizer=tokenizer
)

## 5. Text Generation

Generate text with the trained model using top-k and top-p sampling.

In [ ]:
prompt = "In the beginning"
result = generate_text(
    model=model,
    idx=prompt,
    tokenizer=tokenizer,
    new_max_length=50,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P
)
print("Generated:")
print(result)

## 6. Save Model

In [ ]:
torch.save(model.state_dict(), "model.pt")
print("Model saved to model.pt")

## 7. Load & Continue Training

In [ ]:
# Load a previously trained checkpoint
model = kaitomodel()
model.load_state_dict(torch.load("model.pt", map_location=device))
model.to(device)
print("Checkpoint loaded, continuing training...")

# Continue training for 2 more epochs
train_losses, val_losses, track_tokens_seen = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=2,
    eval_freq=5,
    eval_iter=5,
    start_context="Once upon a time",
    tokenizer=tokenizer
)

## 8. MoE Training (optional)

To train the MoE variant (510M params, sparse activation):
1. Set `USE_MOE = True` in `config.py`
2. Restart the kernel and run cells 1-5 again

Note: MoE requires more VRAM and converges at a different LR schedule.
Consider using a lower learning rate (e.g., `LEARNING_RATE = 1e-4`).